In [9]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================


BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "BPQ_D.xpt"
csv_file = DATA_DIR / "BPQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source copy for final fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 BPQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BPQ020",
    "BPQ030",
    "BPQ040A",
    "BPQ050A",
    "BPQ060",
    "BPQ070",
    "BPQ080",
    "BPQ090A",
    "BPQ090B",
    "BPQ090C",
    "BPQ090D",
    "BPQ100A",
    "BPQ100B",
    "BPQ100C",
    "BPQ100D"
]


if len(df) != 6139:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 16:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "BPQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: BPQ_D structure = "
    "6,139 rows x 16 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- TINY VALUE CHECK ---")

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: BPQ_D unexpectedly contains "
        "XPORT tiny-value artifacts. "
        "Do NOT automatically convert them to zero."
    )


print(
    "PASS: BPQ_D contains no tiny-value artifacts."
)


# ============================================================
# 5. TRUE ZERO CHECK
# ============================================================

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    zero_total
)


if zero_total != 0:

    print(
        zero_counts[
            zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "Unexpected zero value(s) found. "
        "Review against the CDC codebook before proceeding."
    )


print(
    "PASS: Source contains no true numeric zeros."
)


# ============================================================
# 6. CONFIRM NO GENUINE DECIMAL VARIABLES
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_failures = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    print(
        f"{col:<10}: "
        f"fractional values = {count:,}"
    )


    if count != 0:

        fractional_failures.append(
            col
        )


if fractional_failures:

    raise ValueError(
        "STOP: Genuine decimal values found in:\n"
        + str(fractional_failures)
    )


print(
    "PASS: All nonmissing BPQ_D values "
    "are exact whole-number identifiers/codes."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:
    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:
    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31130:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. CDC EXPECTED FREQUENCIES
# ============================================================

#
# Dictionary structure:
#
# counts = nonmissing code frequencies
# missing = expected SAS missing count
#

EXPECTED_FREQUENCIES = {

    "BPQ020": {
        "counts": {
            1: 1625,
            2: 4498,
            9: 16
        },
        "missing": 0
    },

    "BPQ030": {
        "counts": {
            1: 1318,
            2: 299,
            9: 8
        },
        "missing": 4514
    },

    "BPQ040A": {
        "counts": {
            1: 1328,
            2: 292,
            9: 5
        },
        "missing": 4514
    },

    "BPQ050A": {
        "counts": {
            1: 1160,
            2: 167,
            9: 1
        },
        "missing": 4811
    },

    "BPQ060": {
        "counts": {
            1: 3370,
            2: 1419,
            9: 190
        },
        "missing": 1160
    },

    "BPQ070": {
        "counts": {
            1: 2021,
            2: 628,
            3: 464,
            4: 225,
            9: 32
        },
        "missing": 2769
    },

    "BPQ080": {
        "counts": {
            1: 1386,
            2: 1961,
            9: 23
        },
        "missing": 2769
    },

    "BPQ090A": {
        "counts": {
            1: 1074,
            2: 307,
            9: 5
        },
        "missing": 4753
    },

    "BPQ090B": {
        "counts": {
            1: 753,
            2: 628,
            9: 5
        },
        "missing": 4753
    },

    "BPQ090C": {
        "counts": {
            1: 887,
            2: 495,
            9: 4
        },
        "missing": 4753
    },

    "BPQ090D": {
        "counts": {
            1: 851,
            2: 532,
            9: 3
        },
        "missing": 4753
    },

    "BPQ100A": {
        "counts": {
            1: 892,
            2: 182
        },
        "missing": 5065
    },

    "BPQ100B": {
        "counts": {
            1: 645,
            2: 108
        },
        "missing": 5386
    },

    "BPQ100C": {
        "counts": {
            1: 661,
            2: 226
        },
        "missing": 5252
    },

    "BPQ100D": {
        "counts": {
            1: 704,
            2: 146,
            9: 1
        },
        "missing": 5288
    }
}


# ============================================================
# 9. VALIDATE ALL CDC FREQUENCIES
# ============================================================

print("\n--- CDC FREQUENCY CHECKS ---")


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    # Convert float keys such as 1.0 -> int 1
    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CDC frequency mismatch.\n"
            f"Expected: {spec['counts']}\n"
            f"Actual:   {actual_counts}"
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: missing count mismatch.\n"
            f"Expected: {spec['missing']:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col:<8} "
        f"valid={sum(actual_counts.values()):,} "
        f"missing={actual_missing:,}"
    )


print(
    "PASS: All BPQ_D frequencies match CDC."
)


# ============================================================
# 10. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# BPQ_D has:
#
# - no tiny artifacts
# - no zero restorations
# - no recalculation
#
# Therefore expected values are exactly the original XPT.
#

expected_df = source_df.copy()


# ============================================================
# 11. CLEAN SEQN + QUESTIONNAIRE CODE FORMATTING
# ============================================================

#
# We proved every nonmissing value is an exact
# whole number.
#
# Therefore Int64 formatting is appropriate.
#
# Examples:
#
# SEQN:
# 31130.0 -> 31130
#
# BPQ020:
# 1.0 -> 1
#
# BPQ070:
# 4.0 -> 4
#
# Missing:
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and questionnaire fields "
    "converted to clean integer formatting."
)


# ============================================================
# 12. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: No BPQ_D numeric values changed."
)


# ============================================================
# 13. EXPORT CSV
# ============================================================

#
# Do not use float_format.
#
# Int64 ensures questionnaire values do not
# receive unnecessary .0.
#
# Missing values export as blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 14. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 15. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 16. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 16 CSV columns numerically "
    "match the original XPT exactly."
)


# ============================================================
# 17. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    diff = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += diff


    if diff > 0:

        missing_failures.append(
            (
                col,
                diff
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-value positions changed:\n"
        + str(missing_failures)
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 18. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


if len(raw_rows) != 6139:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 19. STRICT RAW FORMAT FIDELITY
# ============================================================

#
# All BPQ_D fields are whole-number identifiers/codes.
#
# Expected token:
#
# ""                if missing
#
# str(int(value))   otherwise
#
# Examples:
#
# 31130.0 -> "31130"
# 1.0     -> "1"
# 9.0     -> "9"
# NaN     -> ""
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(raw_examples) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "Examples:"
    )

    for example in raw_examples:
        print(example)


    raise ValueError(
        "BPQ_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches expected "
    "whole-number questionnaire representation."
)


# ============================================================
# 20. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(dot_zero_examples) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 21. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 22. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 23. FINAL CDC FREQUENCY CHECK AFTER EXPORT
# ============================================================

print("\n--- FINAL CSV FREQUENCY CHECK ---")


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        csv_df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        csv_df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:
        raise ValueError(
            f"{col}: CSV frequencies changed."
        )


    if actual_missing != spec["missing"]:
        raise ValueError(
            f"{col}: CSV missing count changed."
        )


print(
    "PASS: Final CSV frequencies still match CDC."
)


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("BPQ_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Tiny artifacts in XPT:",
    0
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "Genuine decimal variables:",
    0
)

print(
    "SEQN/code formatting:",
    "integer"
)

print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Values ending .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)

print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "BPQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 BPQ_D XPT -> CSV
Rows: 6,139
Columns: 16
PASS: BPQ_D structure = 6,139 rows x 16 columns.

--- TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: BPQ_D contains no tiny-value artifacts.

True numeric zeros: 0
PASS: Source contains no true numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN      : fractional values = 0
BPQ020    : fractional values = 0
BPQ030    : fractional values = 0
BPQ040A   : fractional values = 0
BPQ050A   : fractional values = 0
BPQ060    : fractional values = 0
BPQ070    : fractional values = 0
BPQ080    : fractional values = 0
BPQ090A   : fractional values = 0
BPQ090B   : fractional values = 0
BPQ090C   : fractional values = 0
BPQ090D   : fractional values = 0
BPQ100A   : fractional values = 0
BPQ100B   : fractional values = 0
BPQ100C   : fractional values = 0
BPQ100D   : fractional values = 0
PASS: All nonmissing BPQ_D values are exact whole-number identifiers/codes.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 31130.0
Maximum: 4